# Thu thập và khám phá dữ liệu

**Phụ trách:** B. Bàn giao các bảng dùng chung theo [phân công](../PHAN_CONG.md).

## 1. Nguồn và phạm vi

Ghi nguồn tải, ngày tải, điều khoản sử dụng, phạm vi thời gian, cách ghi nhận và đơn vị quan sát của từng bảng.

Giải thích lý do nhóm chọn dữ liệu tai nạn giao thông và ba bảng hỗ trợ các câu hỏi nghiên cứu ra sao. Ghi công trình công khai liên quan và điều nhóm muốn khảo sát thêm.

In [ ]:
from pathlib import Path

import pandas as pd

# Notebook có thể chạy từ thư mục gốc hoặc từ thư mục notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "environment.yml").is_file():
    raise FileNotFoundError("Mở Jupyter Notebook từ thư mục gốc của dự án.")

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

## 2. Khám phá dữ liệu (mục 3.2 của đề bài)

Khảo sát riêng ba bảng vụ tai nạn, phương tiện và người bị nạn. Đi sâu vào các cột phục vụ Q01–Q06, rồi tóm tắt ngắn các cột còn lại.

### 2.1. Tổng quan và kiểm tra cấu trúc

- Ghi số dòng, số cột của từng **tệp đầy đủ**, đơn vị quan sát và khóa dự kiến. Cell kiểm tra đầu vào bên dưới chỉ đọc mẫu nhỏ.
- Lập từ điển cho từng cột gồm ý nghĩa, kiểu đọc từ CSV, kiểu dùng khi phân tích, đơn vị hoặc bảng mã. Ghi cột cần chuyển kiểu hay bỏ và lý do.
- Đếm dòng trống, dòng trùng toàn bộ và dòng trùng khóa. Phân biệt bản ghi lặp với nhiều phương tiện hoặc nhiều người bị nạn thuộc cùng một vụ. Ghi quyết định giữ hoặc bỏ từng nhóm và số dòng bị ảnh hưởng.

[Đề bài CSC17104](../CSC17104_2026_Final_Project.pdf)

In [ ]:
expected_keys = {
    "Accidents0515.csv": {"Accident_Index"},
    "Vehicles0515.csv": {"Accident_Index", "Vehicle_Reference"},
    "Casualties0515.csv": {"Accident_Index", "Casualty_Reference", "Vehicle_Reference"},
}

for filename, keys in expected_keys.items():
    csv_path = raw_dir / filename
    if not csv_path.is_file():
        print(f"Chưa có {filename}. Tải CSV theo HUONG_DAN.md.")
        continue

    # Giữ mã vụ dưới dạng chuỗi và đọc mẫu nhỏ để tránh nạp cả tệp lớn.
    preview = pd.read_csv(csv_path, nrows=5, dtype={"Accident_Index": "string"})
    missing_keys = sorted(keys - set(preview.columns))
    if missing_keys:
        raise ValueError(f"{filename} thiếu cột {missing_keys}")
    print(f"{filename}: {len(preview.columns)} cột, có đủ cột khóa dự kiến.")

In [ ]:
# Ghi lại shape và dtypes của từng bảng khi đã đọc toàn bộ tệp.
# Dùng duplicated() cho bản ghi trùng hoàn toàn. Kiểm tra khóa riêng
# theo đơn vị quan sát của từng bảng.

### 2.2. Cột định lượng

- Với các cột liên quan, chẳng hạn tuổi người bị nạn hoặc số phương tiện mỗi vụ, mô tả hình dạng phân phối, tâm (trung bình hoặc trung vị) và độ phân tán (độ lệch chuẩn hoặc IQR). Báo cả min/max và kiểm tra xem có hợp lý theo đơn vị và miền giá trị.
- Có thể dùng IQR hoặc z-score để đánh dấu điểm cần xem lại. Đối chiếu giá trị cực đoan với tài liệu và bối cảnh trước khi kết luận đó là bản ghi hợp lệ hay lỗi.
- Kiểm tra giá trị bất khả thi và mã giữ chỗ như tuổi âm, `-1` hoặc `999`. Xác minh ý nghĩa theo từng cột trước khi đổi mã hoặc loại dòng.

In [ ]:
# Chọn các cột đo lường, dùng describe() và vẽ phân phối cho cột liên quan.
# Ghi số dòng bị gắn cờ, lý do và quyết định sau khi kiểm tra ngoại lệ.

### 2.3. Cột phân loại

- Đếm số nhãn phân biệt, báo nhãn phổ biến và tần suất hoặc tỷ lệ. Kiểm tra mất cân bằng ở các nhóm quan trọng như loại đường, ánh sáng, phương tiện hoặc mức độ nghiêm trọng.
- Đối chiếu mã phân loại với từ điển. Kiểm tra khác biệt cách viết, khoảng trắng, lỗi chính tả hoặc nhiều mã tương đương. Cột lưu bằng số vẫn có thể là cột phân loại.
- Liệt kê nhãn hiếm, nêu tiêu chí và lý do nếu gộp nhóm. Báo cỡ mẫu sau khi chuẩn hóa để người đọc thấy tác động.

In [ ]:
# Dùng nunique() và value_counts(dropna=False) để rà số nhãn và tần suất.
# Xác minh ý nghĩa của từng mã trước khi chuẩn hóa hoặc gộp nhóm.

### 2.4. Dữ liệu thiếu

- Lập bảng số lượng và tỷ lệ thiếu **cho từng cột của từng bảng**, dựa trên số dòng tương ứng. Thống kê riêng ô trống và mã không rõ trước khi quy đổi.
- So sánh tỷ lệ thiếu theo năm, loại đường hoặc nhóm đối tượng liên quan. Ghi nguyên nhân có bằng chứng từ nguồn và đánh dấu các giả thuyết chưa xác minh.
- Nêu phương án theo cột, chẳng hạn bỏ dòng, điền giá trị hoặc tách thành nhóm riêng, cùng lý do và cỡ mẫu chịu ảnh hưởng. Với ML, học tham số điền thiếu trong pipeline trên tập huấn luyện.

In [ ]:
# Sau khi xác định mã thiếu, dùng isna().sum() và isna().mean()
# để lập bảng số lượng và tỷ lệ. So sánh độ thiếu giữa các nhóm.

### 2.5. Mối quan hệ giữa các biến

- Xem tương quan giữa những biến định lượng liên quan trên cùng đơn vị quan sát. Ghi số cặp có dữ liệu và giải thích cách chọn thước đo.
- Dùng bảng chéo và thống kê theo nhóm cho các tổ hợp có ý nghĩa, chẳng hạn loại đường với điều kiện ánh sáng, hoặc nhóm tuổi với loại phương tiện. Ghi số quan sát và mẫu số trước khi diễn giải tỷ lệ. Kiểm tra khóa và quan hệ ghép nếu dùng cột từ nhiều bảng.
- Nêu mối liên hệ bất ngờ, kiểm tra nhóm có ít dòng hoặc khác biệt do cơ cấu dữ liệu, rồi giải thích giới hạn diễn giải.

In [ ]:
# Dùng corr() cho biến định lượng, crosstab() hoặc groupby() cho
# tổ hợp phân loại và thống kê theo nhóm. Ghi mẫu số cho từng kết quả.

### 2.6. Tóm tắt khám phá

Viết **3–5 quan sát chính** sau khi chạy trên dữ liệu thật. Chỉ ra các vấn đề chất lượng nghiêm trọng, cách dự kiến tiền xử lý và những mẫu hình có thể dẫn đến hoặc giúp tinh chỉnh câu hỏi nghiên cứu.

## 3. Làm sạch

Thực hiện các quyết định từ phần khám phá. Ghi số dòng trước và sau mỗi thao tác thay đổi số dòng.

In [ ]:
# Viết phần kiểm tra chất lượng và chuẩn hóa từng bảng ở đây.

## 4. Ghép bảng và bàn giao

Ghi khóa ghép, quan hệ kỳ vọng và số dòng chưa ghép được. Bàn giao `accidents_clean.csv`, `casualty_vehicle.csv` và `accident_counts.csv` trong `data/processed/`.

In [ ]:
# Viết phần ghép, kiểm tra đơn vị quan sát và xuất ba bảng ở đây.